In [0]:

# Check available catalogs
display(spark.sql("SHOW CATALOGS"))


catalog
dbw_online_retail_de
online_retail
samples
system


In [0]:

# Check available schemas in the existing catalog
display(spark.sql("SHOW SCHEMAS IN online_retail"))


databaseName
bronze
default
gold
information_schema
silver


In [0]:

# Check existing Unity Catalog external locations

display(spark.sql("SHOW EXTERNAL LOCATIONS"))


name,url,comment
ext_online_retail,abfss://raw@stonlineretailde2026.dfs.core.windows.net/,null


In [0]:

# Display complete external location details

locations = spark.sql("SHOW EXTERNAL LOCATIONS")

locations.show(truncate=False)


+-----------------+------------------------------------------------------+-------+
|name             |url                                                   |comment|
+-----------------+------------------------------------------------------+-------+
|ext_online_retail|abfss://raw@stonlineretailde2026.dfs.core.windows.net/|NULL   |
+-----------------+------------------------------------------------------+-------+



In [0]:

# Check access to the Indian E-Commerce raw data

location_url = locations.first()["url"]

project_path = location_url.rstrip("/") + "/indian_ecommerce/"

print("Testing path:", project_path)

display(dbutils.fs.ls(project_path))


Testing path: abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/


path,name,size,modificationTime
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/customer_reviews.csv,customer_reviews.csv,5325904,1793604278000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/customers.csv,customers.csv,3721855,1793604278000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/marketing_campaigns.csv,marketing_campaigns.csv,23062,1793604252000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/order_items.csv,order_items.csv,17034894,1793604314000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/orders.csv,orders.csv,13542233,1793604308000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/payments.csv,payments.csv,6754058,1793604311000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/products.csv,products.csv,71384,1793604388000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/returns.csv,returns.csv,1070176,1793604288000
abfss://raw@stonlineretailde2026.dfs.core.windows.net/indian_ecommerce/shipments.csv,shipments.csv,9258174,1793604300000


In [0]:

# Create isolated schemas for the Indian E-Commerce project

spark.sql("""
CREATE SCHEMA IF NOT EXISTS online_retail.indian_ecommerce_bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS online_retail.indian_ecommerce_silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS online_retail.indian_ecommerce_gold
""")

print("Indian E-Commerce schemas created successfully!")

display(
    spark.sql("SHOW SCHEMAS IN online_retail")
)


Indian E-Commerce schemas created successfully!


databaseName
bronze
default
gold
indian_ecommerce_bronze
indian_ecommerce_gold
indian_ecommerce_silver
information_schema
silver


In [0]:

# Define Azure Storage source path

RAW_PATH = (
    "abfss://raw@stonlineretailde2026.dfs.core.windows.net/"
    "indian_ecommerce/"
)

# Read orders.csv

orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(RAW_PATH + "orders.csv")
)

print("Orders schema:")
orders_df.printSchema()

print("Sample records:")
display(orders_df.limit(10))

print("Total records:", orders_df.count())


Orders schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_time: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_method: string (nullable = true)
 |-- delivery_city: string (nullable = true)
 |-- delivery_state: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- discount_percentage: string (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- shipping_fee: string (nullable = true)
 |-- tax_amount: string (nullable = true)
 |-- final_amount: string (nullable = true)
 |-- campaign_id: string (nullable = true)

Sample records:


order_id,customer_id,order_date,order_time,order_status,shipping_method,delivery_city,delivery_state,coupon_code,discount_percentage,marketing_channel,subtotal,shipping_fee,tax_amount,final_amount,campaign_id
ORD1000000,CUST103868,2025-08-16,22:35:23,Failed,Next-Day,Mumbai,Maharashtra,FEST20,5.0,SMS Marketing,14439.9,0.0,722.0,15161.9,null
ORD1000001,CUST118472,2023-02-11,18:51:17,Delivered,Economy/Surface,Thiruvananthapuram,Kerala,FEST20,15.0,Social Media - Facebook,4760.0,0.0,238.0,4998.0,CAMP1008
ORD1000002,CUST120793,2024-12-14,11:08:38,Delivered,Economy/Surface,Nagpur,Maharashtra,APP5,30.0,Social Media - Instagram,826.0,0.0,41.3,867.3,CAMP1101
ORD1000003,CUST109166,2025-11-23,09:32:24,Failed,Express,Kochi,Kerala,BIGBILLION,5.0,Direct/App,25878.0,0.0,1293.9,27171.9,null
ORD1000004,CUST119895,2025-07-27,20:35:25,Cancelled,Standard,Jaipur,Rajasthan,SAVE15,30.0,Organic Search,3450.0,0.0,172.5,3622.5,null
ORD1000005,CUST123408,2025-01-08,13:56:23,Delivered,Standard,Coimbatore,Tamil Nadu,BIGBILLION,5.0,Organic Search,10551.1,0.0,527.56,11078.66,null
ORD1000006,CUST117355,2025-05-26,22:26:27,Delivered,Same-Day,Noida,Uttar Pradesh,null,0.0,Referral,15607.3,0.0,780.36,16387.66,CAMP1121
ORD1000007,CUST115056,2025-10-04,11:34:56,Cancelled,Express,Thiruvananthapuram,Kerala,NEW100,30.0,Organic Search,34062.0,0.0,1703.1,35765.1,null
ORD1000008,CUST107870,2023-07-15,15:59:19,Delivered,Standard,Faridabad,Haryana,null,0.0,Social Media - Instagram,35111.2,0.0,1755.56,36866.76,null
ORD1000009,CUST117396,2024-05-26,23:17:31,Failed,Same-Day,Jodhpur,Rajasthan,FLAT50,20.0,Organic Search,2094.4,0.0,104.72,2199.12,null


Total records: 100000


In [0]:

from pyspark.sql.functions import (
    current_timestamp,
    col,
    lit
)
from datetime import datetime, timezone

# Azure Storage source
RAW_PATH = (
    "abfss://raw@stonlineretailde2026.dfs.core.windows.net/"
    "indian_ecommerce/"
)

# Target Unity Catalog schema
TARGET_SCHEMA = "online_retail.indian_ecommerce_bronze"

TABLES = [
    "orders",
    "order_items",
    "customers",
    "products",
    "payments",
    "shipments",
    "returns",
    "customer_reviews",
    "marketing_campaigns"
]

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

ingestion_results = []

for table_name in TABLES:

    print(f"Starting ingestion: {table_name}")

    source_path = f"{RAW_PATH}{table_name}.csv"

    # Read source CSV
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(source_path)
    )

    # Add ingestion metadata using Unity Catalog-compatible syntax
    df = (
        df.withColumn("_source_file", col("_metadata.file_path"))
          .withColumn("_ingestion_timestamp", current_timestamp())
          .withColumn("_batch_id", lit(BATCH_ID))
    )

    # Write to Delta table
    target_table = f"{TARGET_SCHEMA}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    row_count = spark.table(target_table).count()

    ingestion_results.append(
        (table_name, row_count, "SUCCESS")
    )

    print(f"Completed: {table_name} | Rows: {row_count}")

# Display ingestion summary
result_df = spark.createDataFrame(
    ingestion_results,
    ["table_name", "row_count", "status"]
)

display(result_df)

print("BRONZE INGESTION COMPLETED")


Starting ingestion: orders
Completed: orders | Rows: 100000
Starting ingestion: order_items
Completed: order_items | Rows: 254331
Starting ingestion: customers
Completed: customers | Rows: 25000
Starting ingestion: products
Completed: products | Rows: 550
Starting ingestion: payments
Completed: payments | Rows: 100000
Starting ingestion: shipments
Completed: shipments | Rows: 89681
Starting ingestion: returns
Completed: returns | Rows: 12075
Starting ingestion: customer_reviews
Completed: customer_reviews | Rows: 77530
Starting ingestion: marketing_campaigns
Completed: marketing_campaigns | Rows: 151


table_name,row_count,status
orders,100000,SUCCESS
order_items,254331,SUCCESS
customers,25000,SUCCESS
products,550,SUCCESS
payments,100000,SUCCESS
shipments,89681,SUCCESS
returns,12075,SUCCESS
customer_reviews,77530,SUCCESS
marketing_campaigns,151,SUCCESS


BRONZE INGESTION COMPLETED


In [0]:

# Verify Bronze tables

BRONZE_SCHEMA = "online_retail.indian_ecommerce_bronze"

display(
    spark.sql(f"SHOW TABLES IN {BRONZE_SCHEMA}")
)


database,tableName,isTemporary
indian_ecommerce_bronze,customer_reviews,false
indian_ecommerce_bronze,customers,false
indian_ecommerce_bronze,marketing_campaigns,false
indian_ecommerce_bronze,order_items,false
indian_ecommerce_bronze,orders,false
indian_ecommerce_bronze,payments,false
indian_ecommerce_bronze,products,false
indian_ecommerce_bronze,returns,false
indian_ecommerce_bronze,shipments,false


In [0]:

# Verify Delta format and table metadata

for table in [
    "orders",
    "order_items",
    "customers",
    "products",
    "payments",
    "shipments",
    "returns",
    "customer_reviews",
    "marketing_campaigns"
]:
    print(f"\nTable: {table}")

    display(
        spark.sql(
            f"DESCRIBE DETAIL {BRONZE_SCHEMA}.{table}"
        ).select("format", "location")
    )



Table: orders


format,location
delta,



Table: order_items


format,location
delta,



Table: customers


format,location
delta,


... (8 outputs left out)


Table: customer_reviews


format,location
delta,



Table: marketing_campaigns


format,location
delta,
